In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scripts.TPS import ThinPlateSpline
from scripts.plotting import *

# ---------- Load vector field ----------
def load_vector_field(csv_path):
    df = pd.read_csv(csv_path)
    X = df[["x", "y"]].values
    V = df[["vx", "vy"]].values
    time = df["time"].values
    return X, V, time

# ---------- File organization ----------
path_map = {
    "straight_line": "./data/1d/straight_line.csv",
    "sine_curve": "./data/1d/sine_curve.csv",
    "branch_2": "./data/1d/branch_2.csv",
    "branch_4": "./data/1d/branch_4.csv",
    "rotation": "./data/2d/rotation.csv",
    "spiral": "./data/2d/spiral.csv",
    "saddle": "./data/2d/saddle.csv",
    "quadratic_source_sink": "./data/2d/quadratic_source_sink.csv"
}

# Custom layout
row1_names = ["straight_line", "sine_curve", "branch_2", "branch_4"]
row2_names = ["rotation", "spiral", "saddle", "quadratic_source_sink"]
plot_order = row1_names + row2_names

# ---------- 2×4 grid plot ----------
fig, axs = plt.subplots(2, 4, figsize=(20, 10))

for i, (ax, name) in enumerate(zip(axs.ravel(), plot_order)):
    X, V, time = load_vector_field(path_map[name])
    time = (time - np.min(time)) / (np.max(time) - np.min(time))  # normalize time globally

    tps_vf = ThinPlateSpline(X, n_control_points=100)
    tps_vf.fit(V, dof=15)

    if i == 0:
        stream_density = 0.4
        aspect = 2.5
    elif i == 1:
        stream_density = 0.8
        aspect = 2.0
    elif i < 4:
        stream_density = 0.8
        aspect = 1.5
    else:
        stream_density = 1
        aspect = "equal"

    plot_velocity_streamplot(
        X_2d=X,
        tps_vf=tps_vf,
        grid_density=1.0, 
        stream_density=stream_density,
        scatter_color=time,
        scatter_size=40,
        scatter_alpha=0.5,
        ax=ax,
        title=name.replace("_", " "),
        figsize=(5, 4),
        aspect=aspect,
        vmin=0.0,
        vmax=1.0,
        grid_size=50
    )

plt.tight_layout()
plt.show()

In [ ]:
from scripts.pseudotime import *

results = {}

for name in plot_order:
    X, V, time = load_vector_field(path_map[name])
    time = (time - np.min(time)) / (np.max(time) - np.min(time))  # normalize time

    tps_vf = ThinPlateSpline(X, n_control_points=100)
    tps_vf.fit(V, dof=15)

    model = StochasticPseudotime(vector_field=tps_vf.predict, X=X)
    root = model.find_root(n_simulations_per_cell=5)

    if name == "saddle":
        roots = model.find_multiple_roots()
        tau = model.compute_pseudotime(roots=roots[:2])  # use only first 2 roots
    else:
        tau = model.compute_pseudotime(roots=root)

    results[name] = model

In [ ]:
len(paths)

In [ ]:
# --- Load vector field and fit TPS (unchanged) ---
name = "quadratic_source_sink"
X, V, time = load_vector_field(path_map[name])

# --- Select 20 random paths from model.reverse_paths ---
np.random.seed(42)
selected_indices = np.random.choice(len(model.reverse_paths), size=20, replace=False)
paths = [model.reverse_paths[i] for i in selected_indices]

# --- Extract starting points ---
starts = np.array([path[0] for path in paths])

# --- Plot paths and starting points ---
plt.figure(figsize=(8, 8))
for path in paths:
    path_arr = np.array(path)
    plt.plot(path_arr[:, 0], path_arr[:, 1], alpha=0.6)

plt.scatter(X[:, 0], X[:, 1], s=10, alpha=0.4, color='gray', label='Cells')
plt.scatter(starts[:, 0], starts[:, 1], color='red', s=50, label='Start points', zorder=5)
plt.title("MCMC Pseudotime Paths in PCA Space")
plt.axis('equal')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
import numpy as np

row1_names = ["straight_line", "sine_curve", "branch_2", "branch_4"]
row2_names = ["rotation", "spiral", "saddle", "quadratic_source_sink"]
plot_order = row1_names + row2_names

# Define colormap
grey_blue = LinearSegmentedColormap.from_list("greyblue", ["#dddddd", "#1f77b4"])

# Create the grid
fig, axs = plt.subplots(2, 4, figsize=(20, 10))

for i, (ax, name) in enumerate(zip(axs.ravel(), plot_order)):
    model = results[name]
    reverse_paths = model.reverse_paths
    simulated_roots = np.array([path[-1] for path in reverse_paths if len(path) > 0])
    all_points = np.vstack([simulated_roots, model.X])  # Fill sparse bins

    # Plot hexbin
    hb = ax.hexbin(
        all_points[:, 0], all_points[:, 1],
        gridsize=20, cmap=grey_blue, mincnt=1
    )

    # Plot roots
    if name == "saddle":
        multi_roots = model.find_multiple_roots()
        for r in multi_roots[:2]:
            ax.plot(r[0], r[1], 'ro', markersize=6, markeredgecolor='k')
    elif model.root is not None:
        ax.plot(model.root[0], model.root[1], 'ro', markersize=6, markeredgecolor='k')

    ax.set_title(name.replace("_", " ").title(), fontsize=14)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect('equal')
    for spine in ax.spines.values():
        spine.set_visible(False)

# Tweak layout
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Define colormap
viridis = plt.get_cmap("viridis")

# Names and layout
row1_names = ["straight_line", "sine_curve", "branch_2", "branch_4"]
row2_names = ["rotation", "spiral", "saddle", "quadratic_source_sink"]
plot_order = row1_names + row2_names

fig, axs = plt.subplots(2, 4, figsize=(20, 10))

for ax, name in zip(axs.ravel(), plot_order):
    model = results[name]
    tau = model.tau
    X = model.X

    sc = ax.scatter(X[:, 0], X[:, 1], c=tau, cmap=viridis, s=10)
    ax.set_title(name.replace("_", " ").title(), fontsize=14)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect('equal')
    for spine in ax.spines.values():
        spine.set_visible(False)

# Final layout
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

P = model.P

pseudotime_smoothing_width = 1
min_total_mass=1e-6
pseudotime_quantile=0.25
P_smooth = gaussian_filter1d(P, sigma=pseudotime_smoothing_width, axis=1)
total_mass = np.sum(P_smooth, axis=1)
mask = total_mass > min_total_mass

tau = np.full(P.shape[0], np.nan)
cdf = np.cumsum(P_smooth[mask], axis=1) / (total_mass[mask, None] + 1e-10)
tau[mask] = np.argmax(cdf >= pseudotime_quantile, axis=1)


# Pick a few cells to visualize (use cell indices)
selected_cells = [10, 25, 50, 100]  # Change these as you like

# Plot the CDFs
plt.figure(figsize=(8, 6))
timesteps = np.arange(P.shape[1])

for cell in selected_cells:
    if mask[cell]:
        plt.plot(timesteps, cdf[np.where(mask)[0] == cell][0], label=f"Cell {cell}")

plt.xlabel("Timestep")
plt.ylabel("CDF")
plt.title("CDF of Pseudotime Density Across Timesteps")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()